## Before running

In order to keep the GA code consistently usable on Avon, I have created a virtual environment consistent with the python packages on Avon. To use this environment you need to install pipenv with 
- 'pip install pipenv'

Then to create a virtual environment with all the required packages run
- 'pipenv install'

To activate the environment run
- 'pipenv shell'

You can now run the GA code. To exit the virtual environment just use
- 'exit'

All the required packages are listed in 'SOAP_GAS_TMS/Pipfile'

In [ ]:
from genetic_algorithm import *

## Get atomic species

In [ ]:
def get_atomicnumbers(xyz_data):
    atomic_numbers=[]
    for mol in xyz_data:    
        atomic_numbers.extend(mol.get_atomic_numbers())
    return list(set(atomic_numbers))

In [ ]:
path="EXAMPLES/Classification/"
data=pd.read_csv(path+"database.csv")
names=data["Name"].values


moles={}
for name in names:
    moles[name]=ase.io.read(path+"xyz/{}.xyz".format(name))


species=get_atomicnumbers(moles.values())
print("Classification species:",species)

In [ ]:
path="EXAMPLES/Regression/"
data=pd.read_csv(path+"database.csv")
names=data["Name"].values


moles={}
for name in names:
    moles[name]=ase.io.read(path+"xyz/{}.xyz".format(name))


species=get_atomicnumbers(moles.values())
print("Regression species:",species)

## Inputs

Dictionaries are taken as input from a parameter file, they contain the parameters for each soap descriptor

In [ ]:
descDict1 = {'lower': 1, 'upper': 10, 'centres': '{1, 6, 7, 8, 9, 15, 16, 17}',
             'neighbours': '{1, 6, 7, 8, 9, 15, 16, 17}', 'ni_R':1, 'nu_S': 1,
             'mutation_chance': 0.50, 'min_cutoff': 4, 'max_cutoff': 10, 
             'min_sigma': 0.1, 'max_sigma': 0.9, 'message_steps': 8}

descDict2 = {'lower': 1, 'upper': 10, 'centres': '{1, 6, 7, 8, 9, 15, 16, 17}',
             'neighbours': '{1, 6, 7, 8, 9, 15, 16, 17}', 'ni_R':1, 'nu_S': 1,
             'mutation_chance': 0.50, 'min_cutoff': 4, 'max_cutoff': 10, 
             'min_sigma': 0.1, 'max_sigma': 0.9, 'message_steps': 8}

Other parameters are also taken as input. These are automatically checked that the parameters are viable

In [ ]:
num_gens = 5
best_sample, lucky_few, population_size, number_of_children = 4, 2, 12, 4
early_stop = 2
early_number = 3 
min_generations = 5

## GeneParameter

GeneParameter class is created from each descriptor dictionary. 

In [ ]:
params1 = GeneParameters(**descDict1)
params2 = GeneParameters(**descDict2)

In [ ]:
params1

## GeneSet

We can use these classes to create a specific set of parameters that are consistant with these values. This returns a randomly generated GeneSet class

In [ ]:
example_gene_set = params1.make_gene_set()
example_gene_set

We can get the parameters used to create the GeneSet class

In [ ]:
example_gene_set.gene_parameters

We can get a descriptor string to be used as an input for getting SOAPs

In [ ]:
example_gene_set.get_soap_string()

We can also mutate the gene using the mutation chance in the GeneParameters class

In [ ]:
print(f"Before mutation {example_gene_set}")
example_gene_set.mutate_gene()
print(f"After mutation {example_gene_set}")

## Individual

An Individual is made up of a list of GeneSet classes.

In [ ]:
example_gene_set_two = params2.make_gene_set()
gene_set_list = [example_gene_set, example_gene_set_two]
example_individual = Individual(gene_set_list[:1])
example_individual

Getting the score for an indivudual

In [ ]:
example_individual.get_score()
example_individual.score

Breeding two individuals to create a child. Mutation is automatically performed during this

In [ ]:
example_individual_two = Individual(gene_set_list)
print(f"Breeding {example_individual} with {example_individual_two}")
child = breed_individuals(example_individual, example_individual_two)
print(f"Created child {child}")

## Population

A Population is a collection of Individual classes. This can be created using a list of GeneParameter classes

In [ ]:
gene_parameters = [params1, params2]
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)
pop

To initialise the population

In [ ]:
pop.initialise_population()

If you want a way of neatly seeing what individuals are in the population

In [ ]:
pop.print_population()

The next generation can then be generated 

In [ ]:
pop.next_generation()
pop.print_population()

So to run the full GA 

In [ ]:
for _ in range(num_gens):
    pop.next_generation()
pop.print_population()

## BestHistory

BestHistory is a class to store the history and check convergence criteria. So the entire GA can be run, printed, and saved using the following code snippet:

In [ ]:
hist = BestHistory(early_stop, early_number, min_generations)
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)

for gen in range(num_gens):
    if hist.converged:
        break
    print(f"Generation {gen}")
    pop.next_generation()
    hist.append(pop)
    write_to_outfile("-------")

There now exists the entire history of the best Individuals throughout each generation that can be saved and easily accessed. 

In [ ]:
vars(hist)

### Running the GA as a script

To run the GA as a script, you just need an input file. See 'SOAP_GAS_TMS/input.py' for an example of the format. 

Then to run the GA with this input file, run 'python genetic_algorithm.py \<input file name without .py extension>'

Running the script creates two files, out_\<input file name>.txt and history_\<input file name>.pkl

If these files already exist, the old ones will be backed up and new ones created. 

The out file contains a log of everything happening in the GA. 
The history file contains a BestHistory object containing the best individuals throughout the GA.

### Testing

In [1]:
from genetic_algorithm import *
input_parameters = __import__('input')

In [2]:
gene_parameters = [GeneParameters(**params) for params in input_parameters.descList]

In [3]:
example_gene_set = [params.make_gene_set() for params in gene_parameters]

In [4]:
example_gene_set[0].cutoff = 5

In [5]:
example_individual = Individual(example_gene_set)

In [6]:
example_individual.get_score()

Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multiple message passes 4 times
Doing multip

3/3 [==============================] - 0s 8ms/step - loss: 0.8158 - accuracy: 0.6957 - val_loss: 0.8831 - val_accuracy: 0.6250
Epoch 26/200
3/3 [==============================] - 0s 9ms/step - loss: 0.8039 - accuracy: 0.6739 - val_loss: 0.9016 - val_accuracy: 0.5417
Epoch 27/200
3/3 [==============================] - 0s 9ms/step - loss: 0.8104 - accuracy: 0.6630 - val_loss: 0.9084 - val_accuracy: 0.5000
Epoch 28/200
3/3 [==============================] - 0s 9ms/step - loss: 0.7913 - accuracy: 0.7065 - val_loss: 0.8789 - val_accuracy: 0.6667
Epoch 29/200
3/3 [==============================] - 0s 9ms/step - loss: 0.7921 - accuracy: 0.6957 - val_loss: 0.8842 - val_accuracy: 0.6667
Epoch 30/200
3/3 [==============================] - 0s 9ms/step - loss: 0.7822 - accuracy: 0.6957 - val_loss: 0.9630 - val_accuracy: 0.5000
Epoch 31/200
3/3 [==============================] - 0s 9ms/step - loss: 0.7786 - accuracy: 0.6957 - val_loss: 0.9323 - val_accuracy: 0.5000
Epoch 32/200
3/3 [===============

3/3 [==============================] - 0s 8ms/step - loss: 0.4843 - accuracy: 0.8370 - val_loss: 1.1003 - val_accuracy: 0.5417
Epoch 84/200
3/3 [==============================] - 0s 8ms/step - loss: 0.4795 - accuracy: 0.8696 - val_loss: 1.3045 - val_accuracy: 0.4167
Epoch 85/200
3/3 [==============================] - 0s 9ms/step - loss: 0.4660 - accuracy: 0.9022 - val_loss: 1.0752 - val_accuracy: 0.5000
Epoch 86/200
3/3 [==============================] - 0s 1ms/step - loss: 0.4422 - accuracy: 0.8913
Epoch 1/200
3/3 [==============================] - 0s 50ms/step - loss: 1.1174 - accuracy: 0.3548 - val_loss: 1.1123 - val_accuracy: 0.2174
Epoch 2/200
3/3 [==============================] - 0s 8ms/step - loss: 1.0533 - accuracy: 0.5269 - val_loss: 1.1063 - val_accuracy: 0.4783
Epoch 3/200
3/3 [==============================] - 0s 8ms/step - loss: 1.0150 - accuracy: 0.5914 - val_loss: 1.1092 - val_accuracy: 0.4783
Epoch 4/200
3/3 [==============================] - 0s 8ms/step - loss: 0.9878

3/3 [==============================] - 0s 10ms/step - loss: 0.5619 - accuracy: 0.7742 - val_loss: 1.2049 - val_accuracy: 0.5217
Epoch 55/200
3/3 [==============================] - 0s 9ms/step - loss: 0.5538 - accuracy: 0.7742 - val_loss: 1.2039 - val_accuracy: 0.5217
Epoch 56/200
3/3 [==============================] - 0s 9ms/step - loss: 0.5512 - accuracy: 0.7849 - val_loss: 1.1681 - val_accuracy: 0.5652
Epoch 57/200
3/3 [==============================] - 0s 10ms/step - loss: 0.5646 - accuracy: 0.7742 - val_loss: 1.1997 - val_accuracy: 0.4783
Epoch 58/200
3/3 [==============================] - 0s 10ms/step - loss: 0.5453 - accuracy: 0.7634 - val_loss: 1.2429 - val_accuracy: 0.5652
Epoch 59/200
3/3 [==============================] - 0s 9ms/step - loss: 0.5488 - accuracy: 0.7527 - val_loss: 1.2710 - val_accuracy: 0.5217
Epoch 60/200
3/3 [==============================] - 0s 9ms/step - loss: 0.5422 - accuracy: 0.7419 - val_loss: 1.2266 - val_accuracy: 0.4783
Epoch 61/200
3/3 [============

3/3 [==============================] - 0s 8ms/step - loss: 0.5972 - accuracy: 0.7419 - val_loss: 1.2326 - val_accuracy: 0.6957
Epoch 48/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5918 - accuracy: 0.7634 - val_loss: 1.2778 - val_accuracy: 0.6087
Epoch 49/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5774 - accuracy: 0.7634 - val_loss: 1.2965 - val_accuracy: 0.6087
Epoch 50/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5828 - accuracy: 0.7419 - val_loss: 1.2724 - val_accuracy: 0.6522
Epoch 51/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5693 - accuracy: 0.7742 - val_loss: 1.3599 - val_accuracy: 0.5652
Epoch 52/200
3/3 [==============================] - 0s 9ms/step - loss: 0.5614 - accuracy: 0.7957 - val_loss: 1.2876 - val_accuracy: 0.6957
Epoch 53/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5547 - accuracy: 0.7527 - val_loss: 1.3676 - val_accuracy: 0.6087
Epoch 54/200
3/3 [===============

3/3 [==============================] - 0s 8ms/step - loss: 0.7316 - accuracy: 0.6989 - val_loss: 0.9427 - val_accuracy: 0.6522
Epoch 38/200
3/3 [==============================] - 0s 8ms/step - loss: 0.7213 - accuracy: 0.7312 - val_loss: 0.9107 - val_accuracy: 0.5652
Epoch 39/200
3/3 [==============================] - 0s 8ms/step - loss: 0.7147 - accuracy: 0.7312 - val_loss: 0.9267 - val_accuracy: 0.5652
Epoch 40/200
3/3 [==============================] - 0s 8ms/step - loss: 0.7105 - accuracy: 0.7312 - val_loss: 0.9725 - val_accuracy: 0.6957
Epoch 41/200
3/3 [==============================] - 0s 8ms/step - loss: 0.6993 - accuracy: 0.7312 - val_loss: 0.9251 - val_accuracy: 0.5652
Epoch 42/200
3/3 [==============================] - 0s 8ms/step - loss: 0.6927 - accuracy: 0.7419 - val_loss: 0.9044 - val_accuracy: 0.5652
Epoch 43/200
3/3 [==============================] - 0s 9ms/step - loss: 0.6974 - accuracy: 0.6882 - val_loss: 0.9499 - val_accuracy: 0.6522
Epoch 44/200
3/3 [===============

3/3 [==============================] - 0s 8ms/step - loss: 1.0623 - accuracy: 0.5161 - val_loss: 1.0786 - val_accuracy: 0.3913
Epoch 3/200
3/3 [==============================] - 0s 8ms/step - loss: 1.0186 - accuracy: 0.5054 - val_loss: 1.0774 - val_accuracy: 0.3913
Epoch 4/200
3/3 [==============================] - 0s 8ms/step - loss: 0.9952 - accuracy: 0.5054 - val_loss: 1.0890 - val_accuracy: 0.3913
Epoch 5/200
3/3 [==============================] - 0s 9ms/step - loss: 0.9685 - accuracy: 0.5161 - val_loss: 1.0990 - val_accuracy: 0.3478
Epoch 6/200
3/3 [==============================] - 0s 8ms/step - loss: 0.9585 - accuracy: 0.5054 - val_loss: 1.1026 - val_accuracy: 0.3913
Epoch 7/200
3/3 [==============================] - 0s 8ms/step - loss: 0.9471 - accuracy: 0.5269 - val_loss: 1.1076 - val_accuracy: 0.3913
Epoch 8/200
3/3 [==============================] - 0s 8ms/step - loss: 0.9355 - accuracy: 0.5269 - val_loss: 1.0964 - val_accuracy: 0.3913
Epoch 9/200
3/3 [======================

3/3 [==============================] - 0s 8ms/step - loss: 0.5936 - accuracy: 0.7849 - val_loss: 1.0402 - val_accuracy: 0.4783
Epoch 61/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5965 - accuracy: 0.7849 - val_loss: 1.0542 - val_accuracy: 0.4783
Epoch 62/200
3/3 [==============================] - 0s 9ms/step - loss: 0.5857 - accuracy: 0.7634 - val_loss: 1.0756 - val_accuracy: 0.4783
Epoch 63/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5738 - accuracy: 0.8172 - val_loss: 1.0892 - val_accuracy: 0.4783
Epoch 64/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5748 - accuracy: 0.8065 - val_loss: 1.0778 - val_accuracy: 0.4783
Epoch 65/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5658 - accuracy: 0.8065 - val_loss: 1.0741 - val_accuracy: 0.4783
Epoch 66/200
3/3 [==============================] - 0s 8ms/step - loss: 0.5593 - accuracy: 0.7957 - val_loss: 1.0997 - val_accuracy: 0.4783
Epoch 67/200
3/3 [===============

In [7]:
example_individual.score

-0.9626227140426635